# Jev

[Jev](https://docs.typesafe.ai) is TypeSafe's first *System One* model: a model for fast, structured decisions inside software. This notebook uses a few handmade examples to answer:

1. **How does it work?**
2. **Is it any good?**
3. **Is it fast?**
4. **Is it cheap?**
5. **What is it good for, and what isn't it good for?**

## How it works

Jev doesn't generate text. You send it two things:

- **State**: the text (or JSON) to judge.
- **Questions**: typed questions about that state, each answered independently and in parallel.

There are three question types, and each returns probabilities instead of free text:

| Type | Asks | Returns |
| --- | --- | --- |
| **Noul** | a yes/no question | `noul`: probability that the answer is yes |
| **Choice** | pick one option from a set | `choice`, a probability per option, and `confidence` |
| **Score** | rate on ordered levels | `score` (expected level), a probability per level, and `confidence` |

For Choice and Score, `confidence` summarises how peaked the distribution is: 1.0 when all probability sits on one option, 0.0 when it's spread evenly.

Jev takes text only, up to 64k tokens per request (32k for the state plus the longest question).

## Setup

Jev is on OpenRouter as `typesafe/jev-1.13`, but not on the chat completions endpoint: it's a decisions model. Use TypeSafe's SDK (`typesafe-sdk`) pointed at OpenRouter. The key is read from a git-ignored `.env` file next to this notebook (`OPENROUTER_API_KEY=...`).

We pin the versioned model ID so results don't change when the `jev-latest` alias moves.

In [ ]:
import os
import time

import pandas as pd
from dotenv import load_dotenv
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

load_dotenv(".env")
client = TypeSafeClient(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api",
    model="typesafe/jev-1.13",
)

PRICE_PER_INPUT_TOKEN = 0.042 / 1_000_000  # output tokens are free
calls = []  # latency and input tokens of every call, for the speed and cost sections


def ask(state, questions):
    start = time.time()
    response = client.system_one(state, questions)
    calls.append({"seconds": time.time() - start, "input_tokens": response.usage.input_tokens, "questions": len(questions)})
    return response

## The three question types on one example

One call, one state, three questions of different types.

In [ ]:
ticket = (
    "I ordered the blue jacket two weeks ago and it still hasn't arrived. The tracking page has said "
    "'label created' since day one. I need it for a trip on Friday. If it doesn't arrive by then I want my money back."
)

response = ask(
    ticket,
    {
        "wants_refund": Noul(instructions="The customer asks for a refund."),
        "topic": Choice(
            instructions="What is the ticket mainly about?",
            criteria={
                "shipping": "A delayed, lost or damaged delivery",
                "billing": "Charges, invoices or payment problems",
                "product": "A defect or question about the product itself",
            },
        ),
        "urgency": Score(
            instructions="How urgent is this ticket?",
            criteria=["Not urgent", "Somewhat urgent", "Very urgent, has a deadline"],
        ),
    },
)

print("wants_refund:", response.nouls["wants_refund"].noul)
topic = response.choices["topic"]
print("topic:       ", topic.choice, "| confidence", topic.confidence, "|", topic.probabilities)
urgency = response.scores["urgency"]
print("urgency:     ", urgency.score, "| confidence", urgency.confidence, "|", urgency.probabilities)
print("model:", response.model)

## Is it any good?

A few handmade examples per question type, mixing easy cases with ones that trip up keyword matching: sarcasm, negation, mixed feelings, and tickets that fit two categories.

### Noul: is the customer satisfied?

In [ ]:
REVIEWS = [
    ("Absolutely love it, works perfectly.", True),
    ("Oh great, another update that breaks everything. Fantastic.", False),  # sarcasm
    ("Not bad at all, I'd happily buy it again.", True),  # negation
    ("The delivery was late, but the product itself is excellent.", True),  # mixed
    ("Looks nice in the photos. In reality it fell apart after two days.", False),
    ("I wanted to love this, I really did.", False),  # implied
    ("It's fine I guess, nothing special.", None),  # genuinely ambiguous
]

rows = []
for review, expected in REVIEWS:
    p = ask(review, {"satisfied": Noul(instructions="The customer is satisfied with the product.")}).nouls["satisfied"].noul
    rows.append({"review": review, "expected": expected, "p_satisfied": p})

nouls = pd.DataFrame(rows)
nouls["correct"] = nouls.apply(lambda r: None if r["expected"] is None else (r["p_satisfied"] >= 0.5) == r["expected"], axis=1)
nouls

### Choice: route a support ticket

In [ ]:
TEAMS = {
    "billing": "Charges, refunds, invoices or payment methods",
    "shipping": "Delivery status, delays, lost or damaged packages",
    "technical": "The app or website not working, bugs, errors",
    "account": "Login, password, email or profile changes",
}
TICKETS = [
    ("My card was charged twice for the same order.", "billing"),
    ("The app crashes every time I open the settings page.", "technical"),
    ("I can't log in, it says my password is wrong but I just reset it.", "account"),
    ("The box arrived crushed and the mug inside is broken.", "shipping"),
    ("I was charged for a package that never arrived.", "billing"),  # billing or shipping?
    ("How do I change the email address on my profile?", "account"),
]

rows = []
for ticket, expected in TICKETS:
    answer = ask(ticket, {"team": Choice(instructions="Which team should handle this ticket?", criteria=TEAMS)}).choices["team"]
    rows.append({"ticket": ticket, "expected": expected, "choice": answer.choice, "confidence": answer.confidence,
                 **{f"p_{k}": v for k, v in answer.probabilities.items()}})

choices = pd.DataFrame(rows)
choices["correct"] = choices["choice"] == choices["expected"]
choices

### Score: how urgent is the ticket?

In [ ]:
URGENCY = [
    "Not urgent: a question or suggestion",
    "Normal: something is wrong but there is a workaround",
    "High: something important is broken, no workaround",
    "Critical: many users affected, money or data at risk right now",
]
MESSAGES = [
    ("Would be nice if the app had a dark mode.", 0),
    ("Export to PDF fails, but CSV export works so I'm using that for now.", 1),
    ("I can't submit my tax filing and the deadline is tomorrow.", 2),
    ("Checkout is down for all our customers since 10am, we're losing sales every minute.", 3),
    ("Quick question: can I change my delivery address after ordering?", 0),
]

rows = []
for message, expected in MESSAGES:
    answer = ask(message, {"urgency": Score(instructions="How urgent is this support message?", criteria=URGENCY)}).scores["urgency"]
    most_likely = int(max(answer.probabilities, key=answer.probabilities.get))
    rows.append({"message": message, "expected": expected, "most_likely": most_likely, "score": answer.score, "confidence": answer.confidence})

scores = pd.DataFrame(rows)
scores["correct"] = scores["most_likely"] == scores["expected"]
scores

In [ ]:
summary = pd.DataFrame(
    {
        "correct": [nouls["correct"].dropna().sum(), choices["correct"].sum(), scores["correct"].sum()],
        "cases": [nouls["correct"].notna().sum(), len(choices), len(scores)],
    },
    index=["Noul (satisfied)", "Choice (routing)", "Score (urgency)"],
)
summary

### Where it struggles

TypeSafe documents [known weak spots](https://docs.typesafe.ai/model-jaggedness/jev-1.13.md) for `jev-1.13`: counting, arithmetic, date comparison, multi-step indirection, and generating text. Two quick checks, both answerable exactly in code:

In [ ]:
WEAK_SPOTS = [
    (
        "Counting",
        {"items": ["apple", "car", "banana", "chair", "cherry", "lamp", "mango", "pear"]},
        "The list contains exactly 5 fruits.",
        True,  # apple, banana, cherry, mango, pear
    ),
    (
        "Date comparison",
        "Invoice issued: 14 March 2026. Payment received: 2026-02-28.",
        "The payment was received after the invoice was issued.",
        False,  # 28 Feb is before 14 March
    ),
]

rows = []
for name, state, question, expected in WEAK_SPOTS:
    p = ask(state, {"q": Noul(instructions=question)}).nouls["q"].noul
    rows.append({"check": name, "question": question, "expected": expected, "p_yes": p})
pd.DataFrame(rows)

The fix TypeSafe recommends: keep the arithmetic in code and give Jev only the judgment. For counting, ask one question per item and add up the answers yourself:

In [ ]:
items = WEAK_SPOTS[0][1]["items"]
response = ask({"items": items}, {f"item_{i}": Noul(instructions=f"items[{i}] is a fruit.") for i in range(len(items))})
fruit_count = sum(response.nouls[f"item_{i}"].noul >= 0.5 for i in range(len(items)))
print(f"Fruits counted in code: {fruit_count}")

## Is it fast?

Latency of every call made so far:

In [ ]:
calls_df = pd.DataFrame(calls)
print(f"{len(calls_df)} calls: median {calls_df['seconds'].median():.2f}s, p90 {calls_df['seconds'].quantile(0.9):.2f}s, max {calls_df['seconds'].max():.2f}s")

**More questions per call.** Jev evaluates all questions in a request in parallel, so adding questions should cost little extra time.

In [ ]:
review = "Great little café. The coffee was excellent and the staff were friendly, but it was noisy and parking was a nightmare."
ASPECTS = ["coffee", "staff", "noise", "parking", "price", "food", "wifi", "music", "seating", "cleanliness",
           "location", "portion size", "desserts", "waiting time", "decor", "lighting", "bathrooms", "outdoor seating",
           "vegetarian options", "opening hours"]

rows = []
for n in [1, 5, 20]:
    for _ in range(3):
        start = time.time()
        ask(review, {f"q{i}": Noul(instructions=f"The review mentions {aspect}.") for i, aspect in enumerate(ASPECTS[:n])})
        rows.append({"questions": n, "seconds": time.time() - start})

fanout = pd.DataFrame(rows).groupby("questions").median().round(2)
fanout["seconds_per_question"] = (fanout["seconds"] / fanout.index).round(3)
fanout

## Is it cheap?

Jev charges only for input tokens, at \$0.042 per million; output tokens are free. Using the token counts the API reported:

In [ ]:
calls_df = pd.DataFrame(calls)
total_tokens = calls_df["input_tokens"].sum()
per_call = calls_df["input_tokens"].median() * PRICE_PER_INPUT_TOKEN
print(f"{len(calls_df)} calls, {total_tokens:,} input tokens, total cost ${total_tokens * PRICE_PER_INPUT_TOKEN:.5f}")
print(f"Typical call: ~{calls_df['input_tokens'].median():.0f} input tokens = ${per_call:.7f}")
print(f"A million calls like that: ${per_call * 1_000_000:.2f}")

## Takeaways

Numbers from our run; they vary slightly between runs.

**How does it work?** You send a piece of state and typed questions (Noul, Choice, Score); Jev answers each one independently with probabilities, not text. Your code decides what to do with them: thresholds, routing, combining several answers.

**Is it any good?** On these handmade cases, yes: 16 of 17 correct.
- Noul handled sarcasm ("Oh great, another update that breaks everything"), negation ("Not bad at all") and mixed reviews. The genuinely ambiguous "It's fine I guess" landed in the middle at 0.37, which is what you want.
- The one Choice miss, "charged for a package that never arrived", fits both billing and shipping. It was also the only answer with confidence below 1.0 (0.66), so a confidence threshold would have caught it for review.
- The documented weak spots (counting, date comparison) didn't show up on these tiny examples; TypeSafe notes the errors grow with the size of the problem. Their advice still holds: keep counting and arithmetic in code, and ask Jev only for the judgment.

This is a handful of examples, not a benchmark. It shows how Jev behaves, not an accuracy figure for your task.

**Is it fast?** Median 0.30 s per call, p90 under 0.5 s. Adding questions barely changes latency: 20 questions in one call took 0.34 s, about 0.017 s per question.

**Is it cheap?** Only input tokens are billed, at \$0.042 per million. A typical call here was ~350 tokens, about \$0.000015, or roughly \$15 per million calls. This whole notebook cost less than a tenth of a cent.

**What is it good for?**
- Classification, routing and yes/no checks over short text: tickets, reviews, messages, captions, log lines.
- Many decisions about the same text at once (one call, many questions).
- Places where you want a probability to threshold, or a confidence to send uncertain cases to a human.

**What isn't it good for?**
- Generating or extracting free text: use a generative model, then let Jev pick among the candidates.
- Arithmetic, counting, dates and exact numbers: do those in code.
- Images, audio or video: convert them to text first.
- Large states full of irrelevant detail, and questions with double negatives or several hops: filter the state and ask direct questions.